# Análisis reproducible del estudio de accesibilidad web en Europa

Este cuaderno contiene el análisis estadístico reproducible del estudio reflejado en el artículo *Accesibilidad web de servicios digitales de interés público en Europa: asociación con indicadores nacionales y robustez entre herramientas automáticas de evaluación* preparado para *Universal Access in the Information Society*.

El análisis parte del archivo `uais_analysis_data.xlsx`, que contiene:

- Información general de los países e indicadores nacionales.
- Resultados de accesibilidad obtenidos mediante ROAW.
- Resultados de accesibilidad obtenidos mediante Siteimprove.
- Metadatos y fuentes de los indicadores utilizados.

Los resultados derivados se calculan mediante código a partir de los datos originales, evitando cálculos manuales intermedios.

## 1. Carga y validación del conjunto de datos

Se carga el archivo de datos utilizado en el estudio y se comprueba que contiene las cuatro hojas previstas.

In [1]:
from google.colab import files

uploaded = files.upload()

Saving uais_analysis_data.xlsx to uais_analysis_data.xlsx


In [2]:
import pandas as pd

archivo = "uais_analysis_data.xlsx"

xls = pd.ExcelFile(archivo)

print("Hojas disponibles:")
print(xls.sheet_names)

Hojas disponibles:
['country_data', 'ROAW', 'Siteimprove', 'indicators_metadata']


### 1.1. Comprobación de la estructura

Se cargan las cuatro hojas del archivo y se verifica:

- El número de países.
- El número de evaluaciones realizadas con cada herramienta.
- El número de indicadores.
- La ausencia de registros duplicados por país y categoría.

In [3]:
country = pd.read_excel(archivo, sheet_name="country_data")
roaw = pd.read_excel(archivo, sheet_name="ROAW")
siteimprove = pd.read_excel(archivo, sheet_name="Siteimprove")
metadata = pd.read_excel(archivo, sheet_name="indicators_metadata")

print("country_data:", country.shape)
print("ROAW:", roaw.shape)
print("Siteimprove:", siteimprove.shape)
print("indicators_metadata:", metadata.shape)

print("\nDuplicados Country en country_data:",
      country["Country"].duplicated().sum())

print("Duplicados Country + Category en ROAW:",
      roaw.duplicated(subset=["Country", "Category"]).sum())

print("Duplicados Country + Category en Siteimprove:",
      siteimprove.duplicated(subset=["Country", "Category"]).sum())

country_data: (31, 7)
ROAW: (155, 5)
Siteimprove: (155, 5)
indicators_metadata: (4, 5)

Duplicados Country en country_data: 0
Duplicados Country + Category en ROAW: 0
Duplicados Country + Category en Siteimprove: 0


### 1.2. Validación de tipos de datos y valores ausentes

Se comprueban los tipos de datos, la presencia de valores ausentes y la distribución de las evaluaciones entre las cinco categorías de sitios web.

Los valores ausentes se mantienen como tales y no se sustituyen ni estiman.

In [4]:
print("TIPOS DE DATOS\n")

print("country_data:")
print(country.dtypes)

print("\nROAW:")
print(roaw.dtypes)

print("\nSiteimprove:")
print(siteimprove.dtypes)


print("\n\nVALORES AUSENTES\n")

print("country_data:")
print(country.isna().sum())

print("\nROAW:")
print(roaw.isna().sum())

print("\nSiteimprove:")
print(siteimprove.isna().sum())


print("\n\nCATEGORÍAS\n")

print("ROAW:")
print(roaw["Category"].value_counts())

print("\nSiteimprove:")
print(siteimprove["Category"].value_counts())

TIPOS DE DATOS

country_data:
Country              object
Regulatory group     object
M49 region           object
HDI                 float64
SPI                 float64
EGDI                float64
Internet access     float64
dtype: object

ROAW:
Country                    object
Category                   object
URL                        object
Score                     float64
Evaluation date    datetime64[ns]
dtype: object

Siteimprove:
Country                    object
Category                   object
URL                        object
Score                     float64
Evaluation date    datetime64[ns]
dtype: object


VALORES AUSENTES

country_data:
Country             0
Regulatory group    0
M49 region          0
HDI                 0
SPI                 1
EGDI                1
Internet access     0
dtype: int64

ROAW:
Country             0
Category            0
URL                 0
Score              23
Evaluation date    23
dtype: int64

Siteimprove:
Country             0
Cate

### 1.3. Validación de las evaluaciones de accesibilidad

Se comprueba que las puntuaciones se encuentran dentro de las escalas esperadas:

- ROAW: 0–10.
- Siteimprove: 0–100.

Posteriormente se vinculan las evaluaciones de ambas herramientas utilizando el país y la categoría del sitio web como identificadores. También se verifica que las URL correspondientes coincidan entre ambas fuentes.

Para cada sitio web se registra el número de herramientas que proporcionaron una puntuación válida.

In [5]:
print("RANGO DE PUNTUACIONES\n")

print("ROAW:")
print("Mínimo:", roaw["Score"].min())
print("Máximo:", roaw["Score"].max())

print("\nSiteimprove:")
print("Mínimo:", siteimprove["Score"].min())
print("Máximo:", siteimprove["Score"].max())


# Transformamos ROAW de escala 0-10 a escala 0-100
roaw["Score_100"] = roaw["Score"] * 10


# Unimos las dos herramientas por país y categoría
accessibility = roaw.merge(
    siteimprove,
    on=["Country", "Category"],
    how="outer",
    suffixes=("_ROAW", "_Siteimprove"),
    validate="one_to_one"
)


# Comprobamos que las URLs coinciden
accessibility["URL_match"] = (
    accessibility["URL_ROAW"] == accessibility["URL_Siteimprove"]
)

print("\nURLs diferentes entre herramientas:",
      (~accessibility["URL_match"]).sum())


# Contamos cuántas herramientas proporcionaron puntuación
accessibility["n_tools"] = accessibility[
    ["Score_100", "Score_Siteimprove"]
].notna().sum(axis=1)


print("\nNúmero de webs según herramientas disponibles:")
print(accessibility["n_tools"].value_counts().sort_index())


print("\nWebs sin puntuación en ninguna herramienta:")
print(
    accessibility.loc[
        accessibility["n_tools"] == 0,
        ["Country", "Category"]
    ].to_string(index=False)
)

RANGO DE PUNTUACIONES

ROAW:
Mínimo: 2.6
Máximo: 10.0

Siteimprove:
Mínimo: 55.0
Máximo: 100.0

URLs diferentes entre herramientas: 0

Número de webs según herramientas disponibles:
n_tools
0      8
1     42
2    105
Name: count, dtype: int64

Webs sin puntuación en ninguna herramienta:
  Country         Category
    Italy       Local gov.
    Italy     News portals
   Latvia       Local gov.
Lithuania Public transport
    Malta    National gov.
   Norway Public transport
  Romania    National gov.
 Slovakia Public transport


## 2. Cálculo de la puntuación combinada de accesibilidad

Las puntuaciones del ROAW se expresan originalmente en una escala de 0 a 10, por lo que se transforman a una escala de 0 a 100.

Para cada sitio web:

- Si existen puntuaciones de ROAW y Siteimprove, se calcula la media aritmética.
- Si solo existe una de las dos puntuaciones, se utiliza la disponible.
- Si ninguna herramienta proporciona puntuación, el valor se mantiene como ausente (`NaN`).

La transformación a una escala común facilita la comparación numérica, pero no implica equivalencia conceptual entre los sistemas de puntuación de ambas herramientas. La puntuación combinada se utiliza únicamente como una medida sintética y sus resultados se contrastan posteriormente con los obtenidos por ROAW y Siteimprove por separado.

In [6]:
# Puntuación final combinada de accesibilidad
accessibility["Accessibility_score"] = accessibility[
    ["Score_100", "Score_Siteimprove"]
].mean(axis=1, skipna=True)


print("Resumen de la puntuación final:")
print(accessibility["Accessibility_score"].describe())


print("\nPrimeras 10 filas:")
print(
    accessibility[
        [
            "Country",
            "Category",
            "Score_100",
            "Score_Siteimprove",
            "n_tools",
            "Accessibility_score"
        ]
    ].head(10).to_string(index=False)
)


print("\nPuntuaciones finales ausentes:")
print(accessibility["Accessibility_score"].isna().sum())

Resumen de la puntuación final:
count    147.000000
mean      72.023810
std       13.927569
min       32.000000
25%       63.250000
50%       73.000000
75%       81.250000
max      100.000000
Name: Accessibility_score, dtype: float64

Primeras 10 filas:
Country         Category  Score_100  Score_Siteimprove  n_tools  Accessibility_score
Austria       Local gov.       42.0               91.0        2                 66.5
Austria    National gov.       79.0              100.0        2                 89.5
Austria     News portals       41.0               75.0        2                 58.0
Austria Public transport       76.0               85.0        2                 80.5
Austria          Tourism       68.0               89.0        2                 78.5
Belgium       Local gov.       42.0               81.0        2                 61.5
Belgium    National gov.       79.0               87.0        2                 83.0
Belgium     News portals       65.0               81.0        2   

## 3. Construcción del conjunto de datos a nivel de país

Las puntuaciones combinadas de las 155 webs se reorganizan para obtener una fila por país y una columna por categoría de sitio web.

A partir de estas puntuaciones se calculan:

- El número de categorías con una puntuación válida para cada país.
- La puntuación media de accesibilidad del país a partir de las categorías disponibles.
- Una puntuación específica de accesibilidad gubernamental, calculada como la media de las categorías `National gov.` y `Local gov.`.

En el caso de la puntuación gubernamental se exige que ambas categorías dispongan de un valor válido, ya que utilizar una sola de ellas no representaría adecuadamente el promedio entre gobierno nacional y local.

El número de categorías disponibles se conserva para poder identificar los países con datos incompletos y realizar posteriormente análisis de sensibilidad.

In [7]:
# Orden de las cinco categorías analizadas
category_order = [
    "National gov.",
    "Local gov.",
    "Tourism",
    "Public transport",
    "News portals"
]

# Reorganizar las puntuaciones:
# una fila por país y una columna por categoría
scores_by_country = (
    accessibility
    .pivot(
        index="Country",
        columns="Category",
        values="Accessibility_score"
    )
    .reindex(columns=category_order)
    .reset_index()
)

# Número de categorías con puntuación válida en cada país
scores_by_country["n_categories"] = (
    scores_by_country[category_order]
    .notna()
    .sum(axis=1)
)

# Puntuación media global de accesibilidad.
# Se calcula utilizando las categorías disponibles.
scores_by_country["Mean_accessibility"] = (
    scores_by_country[category_order]
    .mean(axis=1, skipna=True)
)

# Puntuación conjunta de gobierno nacional + gobierno local.
# Se exige que ambas puntuaciones estén disponibles.
scores_by_country["Government_accessibility"] = (
    scores_by_country[
        ["National gov.", "Local gov."]
    ]
    .mean(axis=1, skipna=False)
)

# Incorporar los indicadores nacionales y las regiones
analysis_data = country.merge(
    scores_by_country,
    on="Country",
    how="left",
    validate="one_to_one"
)

print("Dimensiones del dataset analítico:")
print(analysis_data.shape)

print("\nNúmero de países según categorías disponibles:")
print(
    analysis_data["n_categories"]
    .value_counts()
    .sort_index()
)

print("\nPaíses con menos de 5 categorías disponibles:")
print(
    analysis_data.loc[
        analysis_data["n_categories"] < 5,
        ["Country", "n_categories", "Mean_accessibility"]
    ].to_string(index=False)
)

print("\nPaíses sin puntuación Government_accessibility:")
print(
    analysis_data.loc[
        analysis_data["Government_accessibility"].isna(),
        ["Country", "National gov.", "Local gov."]
    ].to_string(index=False)
)

print("\nPrimeras filas del dataset analítico:")
print(
    analysis_data[
        [
            "Country",
            "National gov.",
            "Local gov.",
            "Tourism",
            "Public transport",
            "News portals",
            "n_categories",
            "Mean_accessibility",
            "Government_accessibility",
            "HDI",
            "SPI",
            "EGDI",
            "Internet access"
        ]
    ].head().to_string(index=False)
)

Dimensiones del dataset analítico:
(31, 15)

Número de países según categorías disponibles:
n_categories
3     1
4     6
5    24
Name: count, dtype: int64

Países con menos de 5 categorías disponibles:
  Country  n_categories  Mean_accessibility
    Italy             3              68.500
   Latvia             4              77.375
Lithuania             4              77.000
    Malta             4              68.500
   Norway             4              77.875
  Romania             4              67.000
 Slovakia             4              70.875

Países sin puntuación Government_accessibility:
Country  National gov.  Local gov.
  Italy           75.5         NaN
 Latvia           93.0         NaN
  Malta            NaN       100.0
Romania            NaN        49.5

Primeras filas del dataset analítico:
 Country  National gov.  Local gov.  Tourism  Public transport  News portals  n_categories  Mean_accessibility  Government_accessibility   HDI   SPI    EGDI  Internet access
 Austria 

### 3.1. Tratamiento de los datos de accesibilidad ausentes

No todos los sitios web pudieron obtener una puntuación válida de accesibilidad. La puntuación media de cada país se calculó utilizando las categorías disponibles, conservando el número de categorías válidas como indicador de cobertura.

El análisis principal utiliza todos los países con una puntuación media disponible. Para comprobar si los datos ausentes pueden influir en los resultados, se realizará adicionalmente un análisis de sensibilidad restringido a los países con puntuaciones válidas en las cinco categorías.

La puntuación combinada de gobierno nacional y local únicamente se calcula cuando ambas categorías están disponibles.

In [8]:
# Identificar países con las cinco categorías completas
analysis_data["complete_accessibility"] = (
    analysis_data["n_categories"] == 5
)

print("Países con las 5 categorías completas:")
print(analysis_data["complete_accessibility"].sum())

print("\nPaíses con datos incompletos:")
print((~analysis_data["complete_accessibility"]).sum())


# Número de valores válidos por categoría
print("\nNúmero de países con puntuación válida por categoría:")
for category in category_order:
    print(
        f"{category}: "
        f"{analysis_data[category].notna().sum()}"
    )


# Número de países válidos para la puntuación gubernamental
print(
    "\nPaíses con Government_accessibility válida:",
    analysis_data["Government_accessibility"].notna().sum()
)


# Mostrar patrón de datos ausentes
missing_pattern = analysis_data[
    ["Country"] + category_order
].copy()

for category in category_order:
    missing_pattern[category] = (
        missing_pattern[category]
        .isna()
        .map({True: "Missing", False: "Available"})
    )

print("\nPatrón de datos ausentes:")
print(
    missing_pattern.loc[
        analysis_data["n_categories"] < 5
    ].to_string(index=False)
)

Países con las 5 categorías completas:
24

Países con datos incompletos:
7

Número de países con puntuación válida por categoría:
National gov.: 29
Local gov.: 29
Tourism: 31
Public transport: 28
News portals: 30

Países con Government_accessibility válida: 27

Patrón de datos ausentes:
  Country National gov. Local gov.   Tourism Public transport News portals
    Italy     Available    Missing Available        Available      Missing
   Latvia     Available    Missing Available        Available    Available
Lithuania     Available  Available Available          Missing    Available
    Malta       Missing  Available Available        Available    Available
   Norway     Available  Available Available          Missing    Available
  Romania       Missing  Available Available        Available    Available
 Slovakia     Available  Available Available          Missing    Available


## 4. Resultados descriptivos de accesibilidad

Se calculan estadísticos descriptivos para las cinco categorías de sitios web, la puntuación media de accesibilidad por país y la puntuación conjunta de gobierno nacional y local.

Para cada variable se reportan:

- Número de observaciones válidas (`n`).
- Media.
- Desviación estándar.
- Mediana.
- Mínimo.
- Máximo.

Las puntuaciones se expresan en una escala de 0 a 100.

In [9]:
# Variables de accesibilidad que se describirán
descriptive_vars = category_order + [
    "Mean_accessibility",
    "Government_accessibility"
]

# Calcular estadísticos descriptivos
descriptive_table = pd.DataFrame({
    "n": analysis_data[descriptive_vars].count(),
    "Mean": analysis_data[descriptive_vars].mean(),
    "SD": analysis_data[descriptive_vars].std(),
    "Median": analysis_data[descriptive_vars].median(),
    "Min": analysis_data[descriptive_vars].min(),
    "Max": analysis_data[descriptive_vars].max()
})

# Redondear para facilitar la lectura
descriptive_table = descriptive_table.round(2)

print(descriptive_table)

                           n   Mean     SD  Median    Min    Max
National gov.             29  81.93   9.68   83.00  59.50   95.0
Local gov.                29  74.59  14.27   76.50  35.00  100.0
Tourism                   31  68.85  11.44   68.00  50.00   95.0
Public transport          28  67.71  16.69   69.00  32.00  100.0
News portals              30  67.27  11.75   70.75  37.00   84.5
Mean_accessibility        31  72.02   5.34   71.10  64.00   84.6
Government_accessibility  27  78.17   8.98   78.25  58.25   95.5


## 5. Análisis de correlación con indicadores nacionales

Se analiza la asociación entre la puntuación media de accesibilidad de cada país y los cuatro indicadores nacionales seleccionados:

- Human Development Index (HDI).
- Social Progress Index (SPI).
- E-Government Development Index (EGDI).
- Internet access.

Para cada indicador se calculan:

- Coeficiente de correlación de Pearson (`r`).
- Valor `p` asociado.
- Coeficiente de correlación de Spearman (`ρ`).
- Valor `p` asociado.

Posteriormente se aplicará una corrección por comparaciones múltiples mediante el procedimiento de Benjamini-Hochberg para controlar la false discovery rate (FDR).

In [10]:
from scipy.stats import pearsonr, spearmanr

# Indicadores nacionales seleccionados para el análisis principal
indicators = [
    "HDI",
    "SPI",
    "EGDI",
    "Internet access"
]

correlation_results = []

for indicator in indicators:
    # Seleccionar únicamente los casos con datos válidos
    subset = analysis_data[
        ["Mean_accessibility", indicator]
    ].dropna()

    # Correlación de Pearson
    pearson_r, pearson_p = pearsonr(
        subset["Mean_accessibility"],
        subset[indicator]
    )

    # Correlación de Spearman
    spearman_rho, spearman_p = spearmanr(
        subset["Mean_accessibility"],
        subset[indicator]
    )

    correlation_results.append({
        "Indicator": indicator,
        "n": len(subset),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p
    })

correlation_table = pd.DataFrame(correlation_results)

# Redondeo solo para visualización
print(
    correlation_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4
    }).to_string(index=False)
)

      Indicator  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
            HDI 31      0.558     0.0011         0.520      0.0027
            SPI 30      0.573     0.0009         0.560      0.0013
           EGDI 30      0.445     0.0138         0.428      0.0183
Internet access 31      0.366     0.0428         0.313      0.0869


### 5.1. Corrección por comparaciones múltiples

Dado que se realizan varias pruebas de correlación dentro del análisis principal, los valores `p` se ajustan mediante el procedimiento de Benjamini-Hochberg para controlar la false discovery rate (FDR).

La corrección se aplica por separado a los valores `p` de Pearson y Spearman correspondientes a los cuatro indicadores nacionales seleccionados.

In [11]:
from statsmodels.stats.multitest import multipletests

# Corrección FDR para Pearson
_, pearson_q, _, _ = multipletests(
    correlation_table["Pearson_p"],
    alpha=0.05,
    method="fdr_bh"
)

# Corrección FDR para Spearman
_, spearman_q, _, _ = multipletests(
    correlation_table["Spearman_p"],
    alpha=0.05,
    method="fdr_bh"
)

correlation_table["Pearson_q"] = pearson_q
correlation_table["Spearman_q"] = spearman_q

print(
    correlation_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Pearson_q": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4,
        "Spearman_q": 4
    }).to_string(index=False)
)

      Indicator  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  Pearson_q  Spearman_q
            HDI 31      0.558     0.0011         0.520      0.0027     0.0022      0.0055
            SPI 30      0.573     0.0009         0.560      0.0013     0.0022      0.0051
           EGDI 30      0.445     0.0138         0.428      0.0183     0.0184      0.0245
Internet access 31      0.366     0.0428         0.313      0.0869     0.0428      0.0869


### 5.2. Análisis de sensibilidad con casos completos

Para evaluar si los datos ausentes en algunas categorías de accesibilidad influyen en las asociaciones observadas, se repite el análisis principal utilizando únicamente los países con puntuaciones válidas en las cinco categorías de sitios web.

Se comparan los coeficientes obtenidos con los del análisis principal basado en todos los países disponibles.

In [12]:
# Filtrar únicamente países con las cinco categorías completas
complete_cases = analysis_data[
    analysis_data["complete_accessibility"]
].copy()

sensitivity_results = []

for indicator in indicators:
    subset = complete_cases[
        ["Mean_accessibility", indicator]
    ].dropna()

    pearson_r, pearson_p = pearsonr(
        subset["Mean_accessibility"],
        subset[indicator]
    )

    spearman_rho, spearman_p = spearmanr(
        subset["Mean_accessibility"],
        subset[indicator]
    )

    sensitivity_results.append({
        "Indicator": indicator,
        "n": len(subset),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p
    })

sensitivity_table = pd.DataFrame(sensitivity_results)

print(
    sensitivity_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4
    }).to_string(index=False)
)

      Indicator  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
            HDI 24      0.651     0.0006         0.614      0.0014
            SPI 23      0.629     0.0013         0.586      0.0033
           EGDI 23      0.390     0.0656         0.318      0.1390
Internet access 24      0.367     0.0779         0.270      0.2027


In [13]:
# Corrección FDR para el análisis de sensibilidad

_, sensitivity_pearson_q, _, _ = multipletests(
    sensitivity_table["Pearson_p"],
    alpha=0.05,
    method="fdr_bh"
)

_, sensitivity_spearman_q, _, _ = multipletests(
    sensitivity_table["Spearman_p"],
    alpha=0.05,
    method="fdr_bh"
)

sensitivity_table["Pearson_q"] = sensitivity_pearson_q
sensitivity_table["Spearman_q"] = sensitivity_spearman_q

print(
    sensitivity_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Pearson_q": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4,
        "Spearman_q": 4
    }).to_string(index=False)
)

      Indicator  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  Pearson_q  Spearman_q
            HDI 24      0.651     0.0006         0.614      0.0014     0.0023      0.0056
            SPI 23      0.629     0.0013         0.586      0.0033     0.0026      0.0066
           EGDI 23      0.390     0.0656         0.318      0.1390     0.0779      0.1853
Internet access 24      0.367     0.0779         0.270      0.2027     0.0779      0.2027


### 5.3. Comparación entre el análisis principal y el análisis de sensibilidad

Se comparan los coeficientes obtenidos utilizando todos los países disponibles con los obtenidos únicamente a partir de los países con información completa en las cinco categorías de accesibilidad.

El objetivo es evaluar la estabilidad de la magnitud y dirección de las asociaciones, independientemente de los cambios en significación estadística producidos por la reducción del tamaño muestral.

In [14]:
# Comparar los coeficientes del análisis principal y de sensibilidad

comparison_table = correlation_table[
    ["Indicator", "n", "Pearson_r", "Spearman_rho"]
].merge(
    sensitivity_table[
        ["Indicator", "n", "Pearson_r", "Spearman_rho"]
    ],
    on="Indicator",
    suffixes=("_main", "_complete")
)

comparison_table["Delta_Pearson"] = (
    comparison_table["Pearson_r_complete"]
    - comparison_table["Pearson_r_main"]
)

comparison_table["Delta_Spearman"] = (
    comparison_table["Spearman_rho_complete"]
    - comparison_table["Spearman_rho_main"]
)

print(comparison_table.round(3).to_string(index=False))

      Indicator  n_main  Pearson_r_main  Spearman_rho_main  n_complete  Pearson_r_complete  Spearman_rho_complete  Delta_Pearson  Delta_Spearman
            HDI      31           0.558              0.520          24               0.651                  0.614          0.094           0.095
            SPI      30           0.573              0.560          23               0.629                  0.586          0.056           0.026
           EGDI      30           0.445              0.428          23               0.390                  0.318         -0.054          -0.110
Internet access      31           0.366              0.313          24               0.367                  0.270          0.001          -0.043


## 6. Correlaciones por categoría de sitio web

Como análisis secundario, se estudia la asociación entre cada uno de los cuatro indicadores nacionales seleccionados y las cinco categorías de sitios web:

- National government.
- Local government.
- Tourism.
- Public transport.
- News portals.

Para cada combinación se calculan los coeficientes de Pearson y Spearman, junto con sus valores `p`.

Dado que este bloque incluye múltiples comparaciones, los valores `p` se ajustan mediante el procedimiento de Benjamini-Hochberg considerando conjuntamente las 20 correlaciones de cada método.

In [15]:
category_correlation_results = []

for indicator in indicators:
    for category in category_order:

        subset = analysis_data[
            [category, indicator]
        ].dropna()

        pearson_r, pearson_p = pearsonr(
            subset[category],
            subset[indicator]
        )

        spearman_rho, spearman_p = spearmanr(
            subset[category],
            subset[indicator]
        )

        category_correlation_results.append({
            "Indicator": indicator,
            "Category": category,
            "n": len(subset),
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p
        })

category_correlation_table = pd.DataFrame(
    category_correlation_results
)

print(
    category_correlation_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4
    }).to_string(index=False)
)

      Indicator         Category  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
            HDI    National gov. 29      0.381     0.0414         0.353      0.0600
            HDI       Local gov. 29      0.371     0.0476         0.379      0.0424
            HDI          Tourism 31     -0.227     0.2200        -0.179      0.3362
            HDI Public transport 28      0.177     0.3678         0.262      0.1772
            HDI     News portals 30      0.473     0.0082         0.293      0.1156
            SPI    National gov. 28      0.471     0.0115         0.406      0.0320
            SPI       Local gov. 28      0.344     0.0734         0.395      0.0374
            SPI          Tourism 30     -0.261     0.1641        -0.160      0.3972
            SPI Public transport 27      0.170     0.3967         0.235      0.2383
            SPI     News portals 29      0.504     0.0053         0.387      0.0382
           EGDI    National gov. 28      0.385     0.0430         0.430     

### 6.1. Corrección por comparaciones múltiples

Las correlaciones por categoría constituyen un análisis secundario y exploratorio. Dado que se evalúan 20 asociaciones para cada método (4 indicadores × 5 categorías), los valores `p` se ajustan conjuntamente mediante el procedimiento de Benjamini-Hochberg para controlar la false discovery rate (FDR).

In [16]:
# Corrección FDR para las 20 correlaciones de Pearson
_, pearson_category_q, _, _ = multipletests(
    category_correlation_table["Pearson_p"],
    alpha=0.05,
    method="fdr_bh"
)

# Corrección FDR para las 20 correlaciones de Spearman
_, spearman_category_q, _, _ = multipletests(
    category_correlation_table["Spearman_p"],
    alpha=0.05,
    method="fdr_bh"
)

category_correlation_table["Pearson_q"] = pearson_category_q
category_correlation_table["Spearman_q"] = spearman_category_q

print(
    category_correlation_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Pearson_q": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4,
        "Spearman_q": 4
    }).to_string(index=False)
)

      Indicator         Category  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  Pearson_q  Spearman_q
            HDI    National gov. 29      0.381     0.0414         0.353      0.0600     0.1190      0.1714
            HDI       Local gov. 29      0.371     0.0476         0.379      0.0424     0.1190      0.1414
            HDI          Tourism 31     -0.227     0.2200        -0.179      0.3362     0.3385      0.4089
            HDI Public transport 28      0.177     0.3678         0.262      0.1772     0.4542      0.3486
            HDI     News portals 30      0.473     0.0082         0.293      0.1156     0.0575      0.2891
            SPI    National gov. 28      0.471     0.0115         0.406      0.0320     0.0575      0.1414
            SPI       Local gov. 28      0.344     0.0734         0.395      0.0374     0.1630      0.1414
            SPI          Tourism 30     -0.261     0.1641        -0.160      0.3972     0.2735      0.4413
            SPI Public transport 27  

## 7. Asociación entre EGDI y accesibilidad de los sitios gubernamentales

Dada la relación conceptual entre el E-Government Development Index (EGDI) y el desarrollo de los servicios públicos digitales, se realiza un análisis específico de su asociación con una puntuación de accesibilidad gubernamental.

Esta puntuación se calcula como la media de las categorías `National gov.` y `Local gov.` y solo se utiliza cuando ambas puntuaciones están disponibles.

In [17]:
# Seleccionar países con EGDI y puntuación gubernamental disponibles
egdi_gov = analysis_data[
    ["EGDI", "Government_accessibility"]
].dropna()

# Pearson
egdi_gov_pearson_r, egdi_gov_pearson_p = pearsonr(
    egdi_gov["EGDI"],
    egdi_gov["Government_accessibility"]
)

# Spearman
egdi_gov_spearman_rho, egdi_gov_spearman_p = spearmanr(
    egdi_gov["EGDI"],
    egdi_gov["Government_accessibility"]
)

print("n =", len(egdi_gov))
print(
    f"Pearson: r = {egdi_gov_pearson_r:.3f}, "
    f"p = {egdi_gov_pearson_p:.4f}"
)
print(
    f"Spearman: rho = {egdi_gov_spearman_rho:.3f}, "
    f"p = {egdi_gov_spearman_p:.4f}"
)

n = 26
Pearson: r = 0.504, p = 0.0087
Spearman: rho = 0.475, p = 0.0142


### 7.1. Interpretación del análisis específico EGDI–gobierno

La asociación entre EGDI y la puntuación combinada de accesibilidad gubernamental se analiza de forma separada de las correlaciones exploratorias por categoría debido a la relación conceptual específica entre el índice y el desarrollo de los servicios públicos digitales.

La puntuación gubernamental únicamente se calcula cuando ambas categorías están disponibles.

Dado el carácter secundario y conceptualmente motivado de este análisis, los valores p se presentan sin ajuste por comparaciones múltiples y se interpretan de forma exploratoria.

## 8. Diferencias regionales en accesibilidad

Se analiza si la puntuación media de accesibilidad difiere entre las cuatro subregiones europeas definidas por la clasificación M49 de Naciones Unidas:

- Eastern Europe.
- Northern Europe.
- Southern Europe
- Western Europe.

Cyprus se excluye únicamente de este análisis regional, ya que la clasificación M49 lo sitúa en Western Asia.

En primer lugar se calculan estadísticos descriptivos por región. Posteriormente se aplica la prueba no paramétrica de Kruskal-Wallis para evaluar si existen diferencias estadísticamente significativas entre las cuatro regiones.

In [18]:
from scipy.stats import kruskal

# Regiones europeas incluidas en el análisis
european_regions = [
    "Eastern Europe",
    "Northern Europe",
    "Southern Europe",
    "Western Europe"
]

# Excluir Cyprus / Western Asia del análisis regional
regional_data = analysis_data[
    analysis_data["M49 region"].isin(european_regions)
].copy()

print("Número total de países en el análisis regional:")
print(len(regional_data))

print("\nNúmero de países por región:")
print(
    regional_data["M49 region"]
    .value_counts()
    .reindex(european_regions)
)


# Estadísticos descriptivos de la accesibilidad media por región
regional_descriptives = (
    regional_data
    .groupby("M49 region")["Mean_accessibility"]
    .agg(
        n="count",
        Mean="mean",
        SD="std",
        Median="median",
        Min="min",
        Max="max"
    )
    .reindex(european_regions)
    .round(2)
)

print("\nEstadísticos descriptivos por región:")
print(regional_descriptives)


# Preparar los cuatro grupos para Kruskal-Wallis
regional_groups = [
    regional_data.loc[
        regional_data["M49 region"] == region,
        "Mean_accessibility"
    ].dropna()
    for region in european_regions
]

# Prueba de Kruskal-Wallis
kw_H, kw_p = kruskal(*regional_groups)

print("\nKruskal-Wallis para Mean_accessibility:")
print(f"H = {kw_H:.3f}")
print(f"p = {kw_p:.4f}")

Número total de países en el análisis regional:
30

Número de países por región:
M49 region
Eastern Europe      6
Northern Europe    10
Southern Europe     7
Western Europe      7
Name: count, dtype: int64

Estadísticos descriptivos por región:
                  n   Mean    SD  Median   Min    Max
M49 region                                           
Eastern Europe    6  67.65  2.69   67.90  64.5  70.88
Northern Europe  10  75.00  4.69   75.05  69.4  84.60
Southern Europe   7  68.77  3.60   68.50  64.0  74.10
Western Europe    7  75.77  4.41   76.70  68.4  83.00

Kruskal-Wallis para Mean_accessibility:
H = 14.130
p = 0.0027


### 8.1. Comparaciones post hoc entre regiones

Dado que la prueba de Kruskal-Wallis mostró diferencias estadísticamente significativas entre las cuatro regiones, se realizan comparaciones post hoc mediante la prueba de Dunn.

Los valores `p` de las comparaciones por pares se ajustan mediante el procedimiento de Holm para controlar el error asociado a las múltiples comparaciones.

In [19]:
!pip -q install scikit-posthocs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.3 MB/s eta 0:00:00


In [20]:
import scikit_posthocs as sp

# Comparaciones post hoc de Dunn con corrección de Holm
dunn_results = sp.posthoc_dunn(
    regional_data,
    val_col="Mean_accessibility",
    group_col="M49 region",
    p_adjust="holm"
)

# Ordenar filas y columnas según el orden utilizado en el estudio
dunn_results = dunn_results.loc[
    european_regions,
    european_regions
]

print("Dunn post hoc con corrección de Holm:")
print(dunn_results.round(4))

Dunn post hoc con corrección de Holm:
                 Eastern Europe  Northern Europe  Southern Europe  \
Eastern Europe           1.0000           0.0299           1.0000   
Northern Europe          0.0299           1.0000           0.0364   
Southern Europe          1.0000           0.0364           1.0000   
Western Europe           0.0359           1.0000           0.0384   

                 Western Europe  
Eastern Europe           0.0359  
Northern Europe          1.0000  
Southern Europe          0.0384  
Western Europe           1.0000  


In [21]:
# Tamaño del efecto para Kruskal-Wallis: epsilon squared
n_regional = len(regional_data)
k_regional = len(european_regions)

epsilon_squared = (
    (kw_H - k_regional + 1) /
    (n_regional - k_regional)
)

print(f"Epsilon squared = {epsilon_squared:.3f}")

Epsilon squared = 0.428


### 8.2. Diferencias regionales por categoría de sitio web

Como análisis secundario, se evalúa si las diferencias regionales observadas en la puntuación media global se reproducen utilizando la puntuación combinada de accesibilidad de cada una de las cinco categorías de sitios web.

Se aplica una prueba de Kruskal-Wallis independiente para cada categoría. Dado que se realizan cinco contrastes, los valores `p` se ajustan mediante el procedimiento de Benjamini-Hochberg para controlar la false discovery rate (FDR).

In [22]:
regional_category_results = []

for category in category_order:

    # Grupos regionales con datos válidos para esta categoría
    groups = [
        regional_data.loc[
            regional_data["M49 region"] == region,
            category
        ].dropna()
        for region in european_regions
    ]

    # Kruskal-Wallis
    H, p = kruskal(*groups)

    # Número total de observaciones válidas
    n_valid = sum(len(group) for group in groups)

    regional_category_results.append({
        "Category": category,
        "n": n_valid,
        "H": H,
        "p": p
    })

regional_category_table = pd.DataFrame(
    regional_category_results
)

# Corrección FDR para las cinco pruebas
_, regional_category_q, _, _ = multipletests(
    regional_category_table["p"],
    alpha=0.05,
    method="fdr_bh"
)

regional_category_table["q"] = regional_category_q

print(
    regional_category_table.round({
        "H": 3,
        "p": 4,
        "q": 4
    }).to_string(index=False)
)

        Category  n     H      p      q
   National gov. 28 4.082 0.2527 0.4145
      Local gov. 28 4.885 0.1804 0.4145
         Tourism 30 0.906 0.8241 0.8241
Public transport 27 9.652 0.0218 0.1089
    News portals 29 3.417 0.3316 0.4145


### 8.3. Análisis de sensibilidad regional con casos completos

Para comprobar si las diferencias regionales observadas en la puntuación media pueden estar condicionadas por la existencia de países con categorías de accesibilidad ausentes, se repite el análisis regional utilizando únicamente los países con puntuaciones válidas en las cinco categorías.

In [23]:
# Seleccionar únicamente países europeos con las cinco categorías completas
regional_complete = regional_data[
    regional_data["complete_accessibility"]
].copy()

print("Número total de países completos en el análisis regional:")
print(len(regional_complete))

print("\nNúmero de países completos por región:")
print(
    regional_complete["M49 region"]
    .value_counts()
    .reindex(european_regions)
)

# Estadísticos descriptivos
regional_complete_descriptives = (
    regional_complete
    .groupby("M49 region")["Mean_accessibility"]
    .agg(
        n="count",
        Mean="mean",
        SD="std",
        Median="median",
        Min="min",
        Max="max"
    )
    .reindex(european_regions)
    .round(2)
)

print("\nEstadísticos descriptivos:")
print(regional_complete_descriptives)

# Preparar grupos
complete_groups = [
    regional_complete.loc[
        regional_complete["M49 region"] == region,
        "Mean_accessibility"
    ].dropna()
    for region in european_regions
]

# Kruskal-Wallis
kw_complete_H, kw_complete_p = kruskal(*complete_groups)

print("\nKruskal-Wallis con casos completos:")
print(f"H = {kw_complete_H:.3f}")
print(f"p = {kw_complete_p:.4f}")

# Tamaño del efecto
n_complete_regional = len(regional_complete)
k_complete_regional = len(european_regions)

epsilon_complete = (
    (kw_complete_H - k_complete_regional + 1) /
    (n_complete_regional - k_complete_regional)
)

print(f"Epsilon squared = {epsilon_complete:.3f}")

Número total de países completos en el análisis regional:
23

Número de países completos por región:
M49 region
Eastern Europe     4
Northern Europe    7
Southern Europe    5
Western Europe     7
Name: count, dtype: int64

Estadísticos descriptivos:
                 n   Mean    SD  Median   Min   Max
M49 region                                         
Eastern Europe   4  67.00  2.82   66.75  64.5  70.0
Northern Europe  7  73.97  5.36   72.60  69.4  84.6
Southern Europe  5  68.88  4.41   67.40  64.0  74.1
Western Europe   7  75.77  4.41   76.70  68.4  83.0

Kruskal-Wallis con casos completos:
H = 9.706
p = 0.0212
Epsilon squared = 0.353


## 9. Análisis de robustez por herramienta de evaluación

ROAW y Siteimprove se trataron como medidas paralelas de accesibilidad, mientras que la puntuación combinada se utilizó como una medida sintética complementaria. En este bloque se analizan por separado los resultados de ambas herramientas con el objetivo de evaluar la sensibilidad de las conclusiones al instrumento utilizado y la consistencia entre sus puntuaciones.

Dado que las evaluaciones de ROAW y Siteimprove se realizaron en periodos diferentes, las discrepancias observadas pueden reflejar tanto diferencias entre los instrumentos como posibles cambios temporales en los sitios web.

El objetivo no es determinar qué herramienta ofrece mejores resultados, sino comprobar si la dirección y magnitud de las asociaciones principales se mantienen cuando las puntuaciones de accesibilidad se calculan exclusivamente a partir de ROAW o exclusivamente a partir de Siteimprove.

Para cada país se calcula:

- La puntuación media de accesibilidad utilizando únicamente ROAW.
- La puntuación media de accesibilidad utilizando únicamente Siteimprove.

Posteriormente se repiten las correlaciones con HDI, SPI, EGDI e Internet access.

In [24]:
# Construir puntuaciones por país utilizando cada herramienta por separado

# ROAW ya está transformado a escala 0-100 en Score_100
roaw_by_country = (
    accessibility
    .pivot(
        index="Country",
        columns="Category",
        values="Score_100"
    )
    .reindex(columns=category_order)
)

# Siteimprove ya utiliza escala 0-100
siteimprove_by_country = (
    accessibility
    .pivot(
        index="Country",
        columns="Category",
        values="Score_Siteimprove"
    )
    .reindex(columns=category_order)
)

# Media por país utilizando las categorías disponibles
roaw_mean = (
    roaw_by_country
    .mean(axis=1, skipna=True)
    .rename("Mean_ROAW")
)

siteimprove_mean = (
    siteimprove_by_country
    .mean(axis=1, skipna=True)
    .rename("Mean_Siteimprove")
)

# Número de categorías disponibles por herramienta
roaw_n = (
    roaw_by_country
    .notna()
    .sum(axis=1)
    .rename("n_categories_ROAW")
)

siteimprove_n = (
    siteimprove_by_country
    .notna()
    .sum(axis=1)
    .rename("n_categories_Siteimprove")
)

# Incorporar estos resultados al dataset analítico
robustness_data = (
    analysis_data
    .merge(roaw_mean, on="Country", how="left")
    .merge(siteimprove_mean, on="Country", how="left")
    .merge(roaw_n, on="Country", how="left")
    .merge(siteimprove_n, on="Country", how="left")
)

print("Cobertura ROAW:")
print(robustness_data["n_categories_ROAW"].value_counts().sort_index())

print("\nCobertura Siteimprove:")
print(robustness_data["n_categories_Siteimprove"].value_counts().sort_index())

print("\nResumen de las medias por herramienta:")
print(
    robustness_data[
        ["Mean_ROAW", "Mean_Siteimprove"]
    ].describe().round(2)
)

Cobertura ROAW:
n_categories_ROAW
1     1
2     2
3     3
4     7
5    18
Name: count, dtype: int64

Cobertura Siteimprove:
n_categories_Siteimprove
1     1
2     2
3     6
4    13
5     9
Name: count, dtype: int64

Resumen de las medias por herramienta:
       Mean_ROAW  Mean_Siteimprove
count      31.00             31.00
mean       61.74             82.50
std         9.34              4.98
min        39.00             73.50
25%        55.83             79.29
50%        62.60             82.25
75%        66.80             85.20
max        78.80             91.67


### 9.1. Correlaciones principales por herramienta

Se repiten las correlaciones principales utilizando las puntuaciones medias obtenidas exclusivamente mediante ROAW y exclusivamente mediante Siteimprove.

La comparación se centra principalmente en la estabilidad de la dirección y magnitud de los coeficientes.

In [25]:
tool_correlation_results = []

tool_scores = {
    "ROAW": "Mean_ROAW",
    "Siteimprove": "Mean_Siteimprove"
}

for tool_name, score_var in tool_scores.items():
    for indicator in indicators:

        subset = robustness_data[
            [score_var, indicator]
        ].dropna()

        pearson_r, pearson_p = pearsonr(
            subset[score_var],
            subset[indicator]
        )

        spearman_rho, spearman_p = spearmanr(
            subset[score_var],
            subset[indicator]
        )

        tool_correlation_results.append({
            "Tool": tool_name,
            "Indicator": indicator,
            "n": len(subset),
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p
        })

tool_correlation_table = pd.DataFrame(
    tool_correlation_results
)

print(
    tool_correlation_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4
    }).to_string(index=False)
)

       Tool       Indicator  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
       ROAW             HDI 31      0.682     0.0000         0.657      0.0001
       ROAW             SPI 30      0.732     0.0000         0.726      0.0000
       ROAW            EGDI 30      0.589     0.0006         0.547      0.0018
       ROAW Internet access 31      0.571     0.0008         0.554      0.0012
Siteimprove             HDI 31      0.473     0.0072         0.409      0.0224
Siteimprove             SPI 30      0.592     0.0006         0.502      0.0047
Siteimprove            EGDI 30      0.356     0.0538         0.271      0.1479
Siteimprove Internet access 31      0.277     0.1311         0.268      0.1447


### 9.2. Corrección por comparaciones múltiples en los análisis por herramienta

Dado que ROAW y Siteimprove se consideran medidas paralelas de accesibilidad, las asociaciones entre las dos herramientas y los cuatro indicadores nacionales constituyen una familia de ocho contrastes.

Los valores `p` de Pearson y Spearman se ajustan por separado mediante el procedimiento de Benjamini-Hochberg.

In [26]:
# Seleccionar únicamente los resultados específicos de ROAW y Siteimprove
tool_specific = tool_correlation_table[
    tool_correlation_table["Tool"].isin(["ROAW", "Siteimprove"])
].copy()

# FDR para las 8 correlaciones de Pearson
_, tool_pearson_q, _, _ = multipletests(
    tool_specific["Pearson_p"],
    alpha=0.05,
    method="fdr_bh"
)

# FDR para las 8 correlaciones de Spearman
_, tool_spearman_q, _, _ = multipletests(
    tool_specific["Spearman_p"],
    alpha=0.05,
    method="fdr_bh"
)

tool_specific["Pearson_q"] = tool_pearson_q
tool_specific["Spearman_q"] = tool_spearman_q

print(
    tool_specific.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Pearson_q": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4,
        "Spearman_q": 4
    }).to_string(index=False)
)

       Tool       Indicator  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  Pearson_q  Spearman_q
       ROAW             HDI 31      0.682     0.0000         0.657      0.0001     0.0001      0.0002
       ROAW             SPI 30      0.732     0.0000         0.726      0.0000     0.0000      0.0000
       ROAW            EGDI 30      0.589     0.0006         0.547      0.0018     0.0012      0.0035
       ROAW Internet access 31      0.571     0.0008         0.554      0.0012     0.0013      0.0033
Siteimprove             HDI 31      0.473     0.0072         0.409      0.0224     0.0096      0.0299
Siteimprove             SPI 30      0.592     0.0006         0.502      0.0047     0.0012      0.0075
Siteimprove            EGDI 30      0.356     0.0538         0.271      0.1479     0.0614      0.1479
Siteimprove Internet access 31      0.277     0.1311         0.268      0.1447     0.1311      0.1479


### 9.3. Consistencia entre herramientas a nivel de sitio web

Para valorar hasta qué punto ROAW y Siteimprove proporcionan resultados consistentes, se analiza la asociación entre las puntuaciones de ambas herramientas en los sitios web para los que se dispone de las dos evaluaciones.

Este análisis no pretende asumir que ambas herramientas utilizan una metodología equivalente, sino comprobar si tienden a ordenar los sitios web de manera similar.

In [27]:
# Seleccionar únicamente webs evaluadas por ambas herramientas
paired_tools = accessibility[
    ["Country", "Category", "Score_100", "Score_Siteimprove"]
].dropna()

# Pearson
tool_pearson_r, tool_pearson_p = pearsonr(
    paired_tools["Score_100"],
    paired_tools["Score_Siteimprove"]
)

# Spearman
tool_spearman_rho, tool_spearman_p = spearmanr(
    paired_tools["Score_100"],
    paired_tools["Score_Siteimprove"]
)

print("Webs evaluadas por ambas herramientas:", len(paired_tools))

print(
    f"Pearson: r = {tool_pearson_r:.3f}, "
    f"p = {tool_pearson_p:.4g}"
)

print(
    f"Spearman: rho = {tool_spearman_rho:.3f}, "
    f"p = {tool_spearman_p:.4g}"
)

# Diferencia sistemática entre puntuaciones
paired_tools["Difference"] = (
    paired_tools["Score_Siteimprove"]
    - paired_tools["Score_100"]
)

print("\nDiferencia Siteimprove - ROAW:")
print(paired_tools["Difference"].describe().round(2))

Webs evaluadas por ambas herramientas: 105
Pearson: r = 0.352, p = 0.0002306
Spearman: rho = 0.316, p = 0.001041

Diferencia Siteimprove - ROAW:
count    105.00
mean      18.92
std       16.24
min      -23.00
25%        8.00
50%       20.00
75%       31.00
max       50.00
Name: Difference, dtype: float64


### 9.4. Consistencia entre herramientas a nivel de país

Dado que la unidad principal de análisis del estudio es el país, se examina también la asociación entre las puntuaciones medias nacionales obtenidas mediante ROAW y Siteimprove.

Este análisis permite comprobar si las diferencias observadas entre ambas herramientas a nivel de sitio web se mantienen o se reducen al agregar las evaluaciones por país.

In [28]:
country_tool_subset = robustness_data[
    ["Country", "Mean_ROAW", "Mean_Siteimprove"]
].dropna()

country_tool_pearson_r, country_tool_pearson_p = pearsonr(
    country_tool_subset["Mean_ROAW"],
    country_tool_subset["Mean_Siteimprove"]
)

country_tool_spearman_rho, country_tool_spearman_p = spearmanr(
    country_tool_subset["Mean_ROAW"],
    country_tool_subset["Mean_Siteimprove"]
)

print("Países:", len(country_tool_subset))

print(
    f"Pearson: r = {country_tool_pearson_r:.3f}, "
    f"p = {country_tool_pearson_p:.4f}"
)

print(
    f"Spearman: rho = {country_tool_spearman_rho:.3f}, "
    f"p = {country_tool_spearman_p:.4f}"
)

Países: 31
Pearson: r = 0.321, p = 0.0784
Spearman: rho = 0.157, p = 0.3996


### 9.5. Consistencia entre herramientas utilizando evaluaciones emparejadas

Como comprobación adicional, se calculan medias nacionales utilizando exclusivamente las categorías para las que existen puntuaciones válidas en ambas herramientas. De este modo, ROAW y Siteimprove se comparan exactamente sobre el mismo conjunto de sitios web dentro de cada país.

In [29]:
# Solo sitios evaluados por ambas herramientas
paired_country = accessibility[
    ["Country", "Category", "Score_100", "Score_Siteimprove"]
].dropna().copy()

# Medias nacionales sobre exactamente las mismas webs
paired_country_means = (
    paired_country
    .groupby("Country")
    .agg(
        Mean_ROAW_paired=("Score_100", "mean"),
        Mean_Siteimprove_paired=("Score_Siteimprove", "mean"),
        n_paired=("Category", "count")
    )
    .reset_index()
)

# Correlaciones
paired_country_pearson_r, paired_country_pearson_p = pearsonr(
    paired_country_means["Mean_ROAW_paired"],
    paired_country_means["Mean_Siteimprove_paired"]
)

paired_country_spearman_rho, paired_country_spearman_p = spearmanr(
    paired_country_means["Mean_ROAW_paired"],
    paired_country_means["Mean_Siteimprove_paired"]
)

print("Países:", len(paired_country_means))

print("\nNúmero de categorías emparejadas por país:")
print(
    paired_country_means["n_paired"]
    .value_counts()
    .sort_index()
)

print(
    f"\nPearson: r = {paired_country_pearson_r:.3f}, "
    f"p = {paired_country_pearson_p:.4f}"
)

print(
    f"Spearman: rho = {paired_country_spearman_rho:.3f}, "
    f"p = {paired_country_spearman_p:.4f}"
)

Países: 30

Número de categorías emparejadas por país:
n_paired
1     2
2     3
3     9
4    10
5     6
Name: count, dtype: int64

Pearson: r = 0.498, p = 0.0051
Spearman: rho = 0.269, p = 0.1503


### 9.6. Robustez de las diferencias regionales por herramienta

Se repite la comparación regional utilizando por separado las puntuaciones medias nacionales obtenidas mediante ROAW y Siteimprove, con el fin de comprobar si el patrón regional observado con la puntuación combinada es consistente entre ambas herramientas.

In [30]:
regional_tool_results = []

for tool, variable in {
    "ROAW": "Mean_ROAW",
    "Siteimprove": "Mean_Siteimprove"
}.items():

    tool_regional = robustness_data[
        robustness_data["M49 region"].isin(european_regions)
    ][["M49 region", variable]].dropna()

    groups = [
        tool_regional.loc[
            tool_regional["M49 region"] == region,
            variable
        ].dropna()
        for region in european_regions
    ]

    H, p = kruskal(*groups)

    n = len(tool_regional)
    k = len(european_regions)

    epsilon_squared = (H - k + 1) / (n - k)

    regional_tool_results.append({
        "Tool": tool,
        "n": n,
        "H": H,
        "p": p,
        "Epsilon_squared": epsilon_squared
    })

regional_tool_table = pd.DataFrame(regional_tool_results)

print(
    regional_tool_table.round({
        "H": 3,
        "p": 4,
        "Epsilon_squared": 3
    }).to_string(index=False)
)

       Tool  n      H      p  Epsilon_squared
       ROAW 30 11.801 0.0081            0.339
Siteimprove 30  4.519 0.2106            0.058


### 9.7. Robustez de la asociación entre EGDI y accesibilidad gubernamental

Como análisis final de robustez, se examina por separado la asociación entre EGDI y la accesibilidad de los sitios gubernamentales obtenida mediante ROAW y Siteimprove.

Para cada herramienta se calcula la media de las categorías `National gov.` y `Local gov.` únicamente cuando ambas puntuaciones están disponibles.

In [31]:
# Construir puntuación gubernamental por herramienta
gov_roaw = roaw_by_country[
    ["National gov.", "Local gov."]
].mean(axis=1, skipna=False).rename("Government_ROAW")

gov_siteimprove = siteimprove_by_country[
    ["National gov.", "Local gov."]
].mean(axis=1, skipna=False).rename("Government_Siteimprove")

# Incorporar al dataset
gov_robustness = (
    analysis_data
    .merge(gov_roaw, on="Country", how="left")
    .merge(gov_siteimprove, on="Country", how="left")
)

gov_tool_results = []

for tool, variable in {
    "ROAW": "Government_ROAW",
    "Siteimprove": "Government_Siteimprove"
}.items():

    subset = gov_robustness[
        ["EGDI", variable]
    ].dropna()

    pearson_r, pearson_p = pearsonr(
        subset["EGDI"],
        subset[variable]
    )

    spearman_rho, spearman_p = spearmanr(
        subset["EGDI"],
        subset[variable]
    )

    gov_tool_results.append({
        "Tool": tool,
        "n": len(subset),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p
    })

gov_tool_table = pd.DataFrame(gov_tool_results)

print(
    gov_tool_table.round({
        "Pearson_r": 3,
        "Pearson_p": 4,
        "Spearman_rho": 3,
        "Spearman_p": 4
    }).to_string(index=False)
)

       Tool  n  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
       ROAW 23      0.489     0.0179         0.525      0.0101
Siteimprove 19      0.512     0.0251         0.380      0.1084


## 10. Preparación de tablas para el manuscrito

A partir de los análisis anteriores se generan las tablas definitivas utilizadas para la redacción del apartado de resultados.

Las tablas se construyen directamente desde los conjuntos de datos y resultados calculados en el notebook, evitando la introducción manual de valores.

La Tabla 1 del manuscrito corresponde a la relación de países incluidos en el estudio y su clasificación regulatoria y geográfica, por lo que no se genera en este bloque de análisis. A continuación se construyen las Tablas 2–4 a partir de los resultados calculados en el notebook.

### 10.1. Table 2. Estadísticos descriptivos de accesibilidad

In [32]:
table2_frames = []

measures = {
    "ROAW": "Score_100",
    "Siteimprove": "Score_Siteimprove",
    "Combined": "Accessibility_score"
}

for measure, variable in measures.items():

    temp = (
        accessibility
        .groupby("Category")[variable]
        .agg(
            n="count",
            Mean="mean",
            SD="std",
            Median="median",
            Min="min",
            Max="max"
        )
        .reindex(category_order)
        .reset_index()
    )

    temp.insert(0, "Measure", measure)

    table2_frames.append(temp)

table2 = pd.concat(table2_frames, ignore_index=True)

table2_display = table2.copy()

for col in ["Mean", "SD", "Median", "Min", "Max"]:
    table2_display[col] = table2_display[col].round(2)

display(table2_display)

,Measure,Category,n,Mean,SD,Median,Min,Max
0,ROAW,National gov.,26,76.69,14.10,76.50,40.0,100.0
1,ROAW,Local gov.,25,63.24,16.48,72.00,35.0,89.0
2,ROAW,Tourism,28,57.14,11.71,58.00,32.0,82.0
3,ROAW,Public transport,24,57.25,19.31,51.50,30.0,88.0
4,ROAW,News portals,29,59.76,14.60,61.00,26.0,89.0
5,Siteimprove,National gov.,25,88.28,8.28,88.00,69.0,100.0
6,Siteimprove,Local gov.,24,85.08,11.17,85.00,60.0,100.0
7,Siteimprove,Tourism,24,81.54,10.13,81.50,59.0,100.0
8,Siteimprove,Public transport,22,78.77,11.27,78.50,55.0,100.0
9,Siteimprove,News portals,25,78.36,8.05,80.00,58.0,92.0


### 10.2. Table 3. Asociación entre accesibilidad e indicadores nacionales

In [33]:
# Resultados de la puntuación combinada
combined_table3 = correlation_table[
    [
        "Indicator",
        "n",
        "Pearson_r",
        "Pearson_p",
        "Pearson_q",
        "Spearman_rho",
        "Spearman_p",
        "Spearman_q"
    ]
].copy()

combined_table3.insert(0, "Measure", "Combined")


# Resultados por herramienta
tools_table3 = tool_specific[
    [
        "Tool",
        "Indicator",
        "n",
        "Pearson_r",
        "Pearson_p",
        "Pearson_q",
        "Spearman_rho",
        "Spearman_p",
        "Spearman_q"
    ]
].copy()

tools_table3 = tools_table3.rename(
    columns={"Tool": "Measure"}
)


# Unir
table3 = pd.concat(
    [combined_table3, tools_table3],
    ignore_index=True
)

# Orden deseado
measure_order = ["Combined", "ROAW", "Siteimprove"]
indicator_order = ["HDI", "SPI", "EGDI", "Internet access"]

table3["Measure"] = pd.Categorical(
    table3["Measure"],
    categories=measure_order,
    ordered=True
)

table3["Indicator"] = pd.Categorical(
    table3["Indicator"],
    categories=indicator_order,
    ordered=True
)

table3 = (
    table3
    .sort_values(["Measure", "Indicator"])
    .reset_index(drop=True)
)

display(table3.round(4))

,Measure,Indicator,n,Pearson_r,Pearson_p,Pearson_q,Spearman_rho,Spearman_p,Spearman_q
0,Combined,HDI,31,0.5577,0.0011,0.0022,0.5195,0.0027,0.0055
1,Combined,SPI,30,0.5727,0.0009,0.0022,0.5602,0.0013,0.0051
2,Combined,EGDI,30,0.4446,0.0138,0.0184,0.4279,0.0183,0.0245
3,Combined,Internet access,31,0.3661,0.0428,0.0428,0.3125,0.0869,0.0869
4,ROAW,HDI,31,0.6821,0.0000,0.0001,0.6571,0.0001,0.0002
5,ROAW,SPI,30,0.7321,0.0000,0.0000,0.7261,0.0000,0.0000
6,ROAW,EGDI,30,0.5892,0.0006,0.0012,0.5467,0.0018,0.0035
7,ROAW,Internet access,31,0.5706,0.0008,0.0013,0.5535,0.0012,0.0033
8,Siteimprove,HDI,31,0.4728,0.0072,0.0096,0.4087,0.0224,0.0299
9,Siteimprove,SPI,30,0.5916,0.0006,0.0012,0.5023,0.0047,0.0075


In [38]:
def fmt_stat(x):
    """Coeficientes r y rho con tres decimales."""
    if pd.isna(x):
        return ""
    return f"{x:.3f}"


def fmt_prob(x):
    """p y q con tres decimales y <.001 cuando corresponda."""
    if pd.isna(x):
        return ""
    if x < 0.001:
        return "<.001"
    return f"={x:.3f}".replace("0.", ".")


# Construir versión compacta de Table 3
table3_final = table3.copy()

table3_final["Pearson r (p; q)"] = table3_final.apply(
    lambda row:
        f"{fmt_stat(row['Pearson_r'])} "
        f"(p{fmt_prob(row['Pearson_p'])}; "
        f"q{fmt_prob(row['Pearson_q'])})",
    axis=1
)

table3_final["Spearman ρ (p; q)"] = table3_final.apply(
    lambda row:
        f"{fmt_stat(row['Spearman_rho'])} "
        f"(p{fmt_prob(row['Spearman_p'])}; "
        f"q{fmt_prob(row['Spearman_q'])})",
    axis=1
)

table3_final = table3_final[
    [
        "Measure",
        "Indicator",
        "n",
        "Pearson r (p; q)",
        "Spearman ρ (p; q)"
    ]
]

display(table3_final)

,Measure,Indicator,n,Pearson r (p; q),Spearman ρ (p; q)
0,Combined,HDI,31,0.558 (p=.001; q=.002),0.520 (p=.003; q=.005)
1,Combined,SPI,30,0.573 (p<.001; q=.002),0.560 (p=.001; q=.005)
2,Combined,EGDI,30,0.445 (p=.014; q=.018),0.428 (p=.018; q=.024)
3,Combined,Internet access,31,0.366 (p=.043; q=.043),0.313 (p=.087; q=.087)
4,ROAW,HDI,31,0.682 (p<.001; q<.001),0.657 (p<.001; q<.001)
5,ROAW,SPI,30,0.732 (p<.001; q<.001),0.726 (p<.001; q<.001)
6,ROAW,EGDI,30,0.589 (p<.001; q=.001),0.547 (p=.002; q=.004)
7,ROAW,Internet access,31,0.571 (p<.001; q=.001),0.554 (p=.001; q=.003)
8,Siteimprove,HDI,31,0.473 (p=.007; q=.010),0.409 (p=.022; q=.030)
9,Siteimprove,SPI,30,0.592 (p<.001; q=.001),0.502 (p=.005; q=.007)


### 10.3. Table 4. Diferencias regionales de accesibilidad

In [35]:
regional_measures = {
    "Combined": "Mean_accessibility",
    "ROAW": "Mean_ROAW",
    "Siteimprove": "Mean_Siteimprove"
}

table4_desc_frames = []

for measure, variable in regional_measures.items():

    temp = (
        robustness_data[
            robustness_data["M49 region"].isin(european_regions)
        ]
        .groupby("M49 region")[variable]
        .agg(
            n="count",
            Mean="mean",
            SD="std",
            Median="median"
        )
        .reindex(european_regions)
        .reset_index()
    )

    temp.insert(0, "Measure", measure)

    table4_desc_frames.append(temp)

table4a = pd.concat(
    table4_desc_frames,
    ignore_index=True
)

display(table4a.round(2))

,Measure,M49 region,n,Mean,SD,Median
0,Combined,Eastern Europe,6,67.65,2.69,67.90
1,Combined,Northern Europe,10,75.00,4.69,75.05
2,Combined,Southern Europe,7,68.77,3.60,68.50
3,Combined,Western Europe,7,75.77,4.41,76.70
4,ROAW,Eastern Europe,6,55.51,9.16,57.62
5,ROAW,Northern Europe,10,67.56,6.72,65.33
6,ROAW,Southern Europe,7,54.77,8.06,49.80
7,ROAW,Western Europe,7,66.98,6.10,67.60
8,Siteimprove,Eastern Europe,6,79.36,4.61,79.88
9,Siteimprove,Northern Europe,10,83.94,5.48,83.17


In [36]:
# Tamaño del efecto de la puntuación combinada
epsilon_combined = (
    (kw_H - len(european_regions) + 1) /
    (len(regional_data) - len(european_regions))
)

combined_regional_result = pd.DataFrame({
    "Measure": ["Combined"],
    "n": [len(regional_data)],
    "H": [kw_H],
    "p": [kw_p],
    "Epsilon_squared": [epsilon_combined]
})

# ROAW y Siteimprove
tools_regional_result = regional_tool_table.copy()

tools_regional_result = tools_regional_result.rename(
    columns={"Tool": "Measure"}
)

table4b = pd.concat(
    [combined_regional_result, tools_regional_result],
    ignore_index=True
)

display(table4b.round(4))

,Measure,n,H,p,Epsilon_squared
0,Combined,30,14.1301,0.0027,0.4281
1,ROAW,30,11.8012,0.0081,0.3385
2,Siteimprove,30,4.5192,0.2106,0.0584


### 10.4. Exportación de las tablas a Excel

In [37]:
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

output_file = results_dir / "UAIS_final_tables.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # Table 2
    table2.to_excel(
        writer,
        sheet_name="Table2_descriptives",
        index=False
    )

    # Table 3: datos completos
    table3.to_excel(
        writer,
        sheet_name="Table3_correlations_raw",
        index=False
    )

    # Table 3: versión preparada para el manuscrito
    table3_final.to_excel(
        writer,
        sheet_name="Table3_manuscript",
        index=False
    )

    # Table 4 - Panel A
    table4a.to_excel(
        writer,
        sheet_name="Table4A_regions_desc",
        index=False
    )

    # Table 4 - Panel B
    table4b.to_excel(
        writer,
        sheet_name="Table4B_regions_tests",
        index=False
    )

print(f"Archivo generado: {output_file}")

Archivo generado: results/UAIS_final_tables_v2.xlsx
